In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types  import *
import sys
sys.path.append("/Workspace/Users/sivana9908_gmail.com#ext#@sivana9908gmail.onmicrosoft.com/Uber-Eats-End-to-End-_Azure-Data-Engineering-Project")
from src.common.spark_utils import standardize_columns
from delta.tables import DeltaTable

In [0]:
#Reading data from adls 
df_orders = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://bronze@ubereaststorage.dfs.core.windows.net/sql/orders/")
)
display(df_orders)


In [0]:
# standardize columns
df_orders= standardize_columns(df_orders)

# standardize data types
df_orders = df_orders.withColumn("subtotal", col("subtotal").cast("int"))

#handle null values
df_orders = df_orders.filter(col("restaurant_id").isNotNull())\
                      .filter(col("customer_id").isNotNull())

#handle duplicates
df_orders= df_orders.dropDuplicates(["order_id"])

#apply business rules
df_orders= df_orders.filter(col("subtotal")>0)\
                     .filter(col("total_amount")>0)\
                         .filter(col("order_date").isNotNull())\
                             .filter(col("order_status").isin("DELIVERED","CANCELLED","PENDING","PREPARING","CONFIRMED"))

#validation
duplication_count  = df_orders.groupBy("order_id").count().filter(col("count")>1).count()
print("duplicate orders",duplication_count)
price_notnull = df_orders.filter(col("subtotal").isNull()).count()
print("price null",price_notnull)
invalid_orders_status= df_orders.filter(~col("order_status").isin("DELIVERED","CANCELLED","PENDING","PREPARING","CONFIRMED")).count()
print("invalid orders status",invalid_orders_status)



In [0]:
table_name= "ubereats_databricks.silver.silver_orders"

if not spark.catalog.tableExists(table_name):

     df_orders.write.format("delta").mode("overwrite").saveAsTable(table_name)   
      
else:

    target = DeltaTable.forName(spark, table_name)
    target.alias("t") \
        .merge(
            df_orders.alias("s"),
            "t.order_id = s.order_id")\
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
